[← Lekcja 2: Prędkość](https://colab.research.google.com/github/InteraktywnaFizykaZPytonem/Interaktywna-Fizyka-z-Pytonem/blob/main/Kinematyka_2_predkosc.ipynb) &nbsp;·&nbsp; [📚 Spis lekcji](https://github.com/InteraktywnaFizykaZPytonem/Interaktywna-Fizyka-z-Pytonem#spis-lekcji) &nbsp;·&nbsp; **Lekcja 3 z 3** (następne wkrótce)

# 🐝 Kinematyka – część 3: Pszczoła przyspiesza (przyspieszenie)

W części 2 pszczoła leciała ze **stałą** prędkością. Ale prędkość też może się zmieniać: pszczoła może przyspieszać, hamować albo skręcać. To, jak szybko zmienia się prędkość, opisuje **przyspieszenie**. W tej lekcji:

1. zdefiniujemy przyspieszenie jako wektor (na wykresie **żółty**),
2. poznamy wzory na ruch ze stałym przyspieszeniem,
3. zobaczymy, jak lot pszczoły zależy od parametrów,
4. rozwiążemy zadanie: **pszczoła musi wlecieć do okna powoli ruszającego pociągu**. 🚆

> 💾 Chcesz zachować swoje zmiany? Wybierz *Plik → Zapisz kopię na Dysku*.

> ⚠️ **Zobaczyłeś błąd `NameError: name '...' is not defined`?** To znaczy, że pominąłeś jakąś komórkę wyżej albo Colab zapomniał zmienne po przerwie. Uruchom komórki po kolei od góry albo wybierz *Środowisko wykonawcze → Uruchom wszystko*.

Komórki uruchamiaj **po kolei od góry**. Zacznij od kodu pomocniczego (rysowanie, animacje i wykresy):

In [ ]:
#@title 🔧 Kod pomocniczy – rysowanie (uruchom i nie przejmuj się nim, wrócimy do niego później)
import numpy as np
import plotly.graph_objects as go

R_PSZCZOLY = 0.25   # promień pszczoły [m]

# kamera: widok domyślny i widoki z różnych stron
WIDOKI = {
    "↺ Reset widoku": dict(eye=dict(x=1.6, y=-1.6, z=1.0), up=dict(x=0, y=0, z=1)),
    "Z góry":        dict(eye=dict(x=0.0, y=0.001, z=2.6), up=dict(x=0, y=1, z=0)),
    "Z przodu":      dict(eye=dict(x=0.0, y=-2.6, z=0.0), up=dict(x=0, y=0, z=1)),
    "Z boku":        dict(eye=dict(x=2.6, y=0.0, z=0.0), up=dict(x=0, y=0, z=1)),
}

def pszczola(p, R=R_PSZCZOLY):
    """Kula w żółto-czarne paski ze środkiem w punkcie p."""
    x, y, z = p
    u, v = np.meshgrid(np.linspace(0, 2*np.pi, 40), np.linspace(0, np.pi, 40))
    return go.Surface(
        x=x + R*np.cos(u)*np.sin(v), y=y + R*np.sin(u)*np.sin(v), z=z + R*np.cos(v),
        surfacecolor=(np.floor(v/(np.pi/7)) % 2),
        colorscale=[[0, "#FFD400"], [0.5, "#FFD400"], [0.5, "#1a1a1a"], [1, "#1a1a1a"]],
        cmin=0, cmax=1, showscale=False, hoverinfo="skip",
        lighting=dict(ambient=0.9, diffuse=0.3, specular=0.05),
    )

def strzalka(poczatek, w, kolor, dl_grotu=0.6, r_grotu=0.18):
    """Strzałka-wektor: trzon (linia) + grot (stożek z trójkątów). Czubek leży w punkcie poczatek + w."""
    poczatek = np.asarray(poczatek, float)
    w = np.asarray(w, float)
    d = np.linalg.norm(w)
    e = w / d if d > 1e-9 else np.array([0, 0, 1.0])
    dl = min(dl_grotu, d)                          # krótkie wektory dostają mniejszy grot
    r = r_grotu * dl / dl_grotu
    ostrze = poczatek + w
    podstawa = ostrze - e * dl
    trzon = go.Scatter3d(x=[poczatek[0], podstawa[0]], y=[poczatek[1], podstawa[1]],
                         z=[poczatek[2], podstawa[2]], mode="lines",
                         line=dict(color=kolor, width=7), hoverinfo="skip")
    a = np.cross(e, [1, 0, 0]) if abs(e[0]) < 0.9 else np.cross(e, [0, 1, 0])
    a = a / np.linalg.norm(a)
    b = np.cross(e, a)
    t = np.linspace(0, 2*np.pi, 16, endpoint=False)
    kolo = podstawa + r*(np.outer(np.cos(t), a) + np.outer(np.sin(t), b))
    punkty = np.vstack([ostrze, kolo, podstawa])   # 0 = czubek, 1..16 = okrąg, 17 = środek podstawy
    n = len(t)
    boki = [(0, 1 + k, 1 + (k + 1) % n) for k in range(n)]
    spod = [(n + 1, 1 + (k + 1) % n, 1 + k) for k in range(n)]
    ij = np.array(boki + spod)
    grot = go.Mesh3d(x=punkty[:, 0], y=punkty[:, 1], z=punkty[:, 2],
                     i=ij[:, 0], j=ij[:, 1], k=ij[:, 2], color=kolor, flatshading=True,
                     lighting=dict(ambient=0.8, diffuse=0.4), hoverinfo="skip")
    return [trzon, grot]

def osie_ukladu(L):
    return go.Scatter3d(x=[-L, L, None, 0, 0, None, 0, 0], y=[0, 0, None, -L, L, None, 0, 0],
                        z=[0, 0, None, 0, 0, None, -L, L], mode="lines",
                        line=dict(color="gray", width=3), hoverinfo="skip")

def menu_widokow(y):
    return dict(type="buttons", direction="right", x=0.0, xanchor="left", y=y, yanchor="bottom",
                buttons=[dict(label=n, method="relayout", args=[{"scene.camera": k}])
                         for n, k in WIDOKI.items()])

def menu_przelacznikow(grupy, y):
    """Dla każdej grupy dwa przyciski: pokaż / ukryj. Grupy działają niezależnie od siebie,
    więc można ukryć kilka elementów naraz.
    grupy = [(nazwa, [indeksy śladów], zmiana_układu_po_pokazaniu, zmiana_układu_po_ukryciu), ...]"""
    menu = []
    for k, (nazwa, indeksy, po_pokazaniu, po_ukryciu) in enumerate(grupy):
        menu.append(dict(
            type="buttons", direction="right", x=0.25*k, xanchor="left", y=y, yanchor="bottom",
            showactive=True, active=0,
            buttons=[dict(label=f"{nazwa}: pokaż", method="update", args=[{"visible": True}, po_pokazaniu, indeksy]),
                     dict(label="ukryj", method="update", args=[{"visible": False}, po_ukryciu, indeksy])]))
    return menu

def zrob_uklad(tytul, zakres, dtick, adnotacje=(), grupy=(), dodatkowe_menu=(), sliders=(),
               gora=150, dol=0, wysokosc=720):
    def os_(nazwa):
        return dict(title=nazwa, range=[-zakres, zakres], dtick=dtick)
    wys_wykresu = wysokosc - gora - dol
    menu = [menu_widokow(1 + 40/wys_wykresu)] + menu_przelacznikow(grupy, 1.0) + list(dodatkowe_menu)
    return go.Layout(
        title=dict(text=tytul, x=0.5, y=0.99, yanchor="top"),
        scene=dict(xaxis=os_("x [m]"), yaxis=os_("y [m]"), zaxis=os_("z [m]"), aspectmode="cube",
                   camera=WIDOKI["↺ Reset widoku"], annotations=list(adnotacje)),
        height=wysokosc, margin=dict(l=0, r=0, t=gora, b=dol), showlegend=False,
        updatemenus=menu, sliders=list(sliders),
        uirevision="stala",   # zachowuje obrót wykresu podczas animacji i zmian
    )

from plotly.subplots import make_subplots

ZOLTY = "#F2B600"       # kolor wektora przyspieszenia
V_MAX_PSZCZOLY = 4.0    # limit prędkości początkowej pszczoły [m/s]
SZER_OKNA = 1.2         # szerokość okna [m]
DL_POCIAGU = 12.0       # długość wagonu [m]
POL_OKNA = 5.0          # odległość lewego brzegu okna od tyłu wagonu [m]
T_MAX = 15.0            # najdłuższy symulowany czas lotu [s]

def kontrolki_animacji(czasy, czas_klatki, redraw, odstep):
    """Suwak czasu oraz przyciski Start / Pauza."""
    suwak = dict(active=0, x=0.12, len=0.80, xanchor="left", y=0, yanchor="top",
                 pad=dict(t=odstep, b=10), font=dict(color="rgba(0,0,0,0)"),
                 currentvalue=dict(prefix="t = ", suffix=" s", font=dict(size=16)),
                 steps=[dict(method="animate", label=f"{t:.1f}",
                             args=[[str(k)], dict(mode="immediate",
                                                  frame=dict(duration=0, redraw=redraw),
                                                  transition=dict(duration=0))])
                        for k, t in enumerate(czasy)])
    odtwarzanie = dict(type="buttons", direction="left", x=0.0, xanchor="left", y=0, yanchor="top",
                       pad=dict(t=odstep), showactive=False, buttons=[
        dict(label="▶ Start", method="animate",
             args=[None, dict(frame=dict(duration=czas_klatki, redraw=redraw), fromcurrent=True,
                              transition=dict(duration=0))]),
        dict(label="⏸ Pauza", method="animate",
             args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate",
                                transition=dict(duration=0))])])
    return suwak, odtwarzanie

# ---------------------------------------------------------------------
# CZĘŚĆ 1: pszczoła w 3D z wektorem przyspieszenia
# ---------------------------------------------------------------------
def lot_z_przyspieszeniem(r0, v0, a, T):
    """Animacja lotu pszczoły ze stałym przyspieszeniem a. Położenie: r0 + v0*t + a*t^2/2."""
    r0 = np.asarray(r0, float); v0 = np.asarray(v0, float); a = np.asarray(a, float)
    if T <= 0:
        print("⚠️ Czas T musi być większy od zera.")
        return
    N = 40
    czasy = np.linspace(0, T, N + 1)
    polozenie = lambda t: r0 + v0*t + a*t**2/2
    predkosc = lambda t: v0 + a*t
    tor = np.array([polozenie(t) for t in np.linspace(0, T, 120)])

    v_max = max(np.linalg.norm(predkosc(t)) for t in czasy)
    a_war = np.linalg.norm(a)
    sv = 1.0 if v_max <= 3 else 3/v_max            # skala strzałki prędkości [m na m/s]
    sa = 2/a_war if a_war > 1e-9 else 1.0          # skala strzałki przyspieszenia [m na m/s²]
    zakres = int(np.ceil(max(6, np.abs(tor).max() + 1)))
    zakres += zakres % 2

    def ruchome(t):   # 0 pszczoła | 1-2 wektor r | 3-4 wektor v | 5-6 wektor a
        p = polozenie(t)
        return ([pszczola(p)] + strzalka([0, 0, 0], p, "red")
                + strzalka(p, sv*predkosc(t), "royalblue") + strzalka(p, sa*a, ZOLTY))

    stale = [
        go.Scatter3d(x=tor[:, 0], y=tor[:, 1], z=tor[:, 2], mode="lines",
                     line=dict(color="gray", width=3, dash="dash"), hoverinfo="skip"),      # 7 tor
        go.Scatter3d(x=[tor[0, 0], tor[-1, 0]], y=[tor[0, 1], tor[-1, 1]], z=[tor[0, 2], tor[-1, 2]],
                     mode="markers+text", marker=dict(size=5, color="gray"),
                     text=["start", "koniec"], textposition="top center", hoverinfo="skip"),  # 8 znaczniki
        osie_ukladu(zakres),                                                                  # 9 osie
    ]
    dane = ruchome(0) + stale
    ramki = [go.Frame(data=ruchome(t), traces=[0, 1, 2, 3, 4, 5, 6], name=str(k))
             for k, t in enumerate(czasy)]

    suwak, odtwarzanie = kontrolki_animacji(czasy, int(1000*min(T, 8)/N), True, 10)
    grupy = [("Pszczoła", [0], {}, {}), ("Wektor r", [1, 2], {}, {}),
             ("Wektor v", [3, 4], {}, {}), ("Wektor a", [5, 6], {}, {})]
    tytul = (f"<b>a</b> = ({a[0]:g}; {a[1]:g}; {a[2]:g}) m/s²,   "
             f"<b>v</b><sub>0</sub> = ({v0[0]:g}; {v0[1]:g}; {v0[2]:g}) m/s,   T = {T:g} s"
             "<br><sup><span style=\"color:red\"><b>r</b></span> – położenie,   "
             f"<span style=\"color:royalblue\"><b>v</b></span> – prędkość (1 m/s = {sv:.2g} m),   "
             f"<span style=\"color:{ZOLTY}\"><b>a</b></span> – przyspieszenie (1 m/s² = {sa:.2g} m)</sup>")
    uklad = zrob_uklad(tytul, zakres, 2, grupy=grupy, dodatkowe_menu=[odtwarzanie],
                       sliders=[suwak], gora=150, dol=110)

    print(f"Po czasie T = {T:g} s:  r = {np.round(polozenie(T), 2)} m,  v = {np.round(predkosc(T), 2)} m/s")
    go.Figure(data=dane, layout=uklad, frames=ramki).show()

# ---------------------------------------------------------------------
# CZĘŚĆ 2: pszczoła i okno pociągu (2D, widok z góry)
# ---------------------------------------------------------------------
def strzalka_2d(p0, w, d_grotu=0.5, kat=25):
    """Współrzędne łamanej rysującej strzałkę 2D od p0 do p0 + w."""
    p0 = np.asarray(p0, float); w = np.asarray(w, float)
    d = np.linalg.norm(w)
    if d < 1e-9:
        return [p0[0]], [p0[1]]
    e = w / d
    ostrze = p0 + w
    dl = min(d_grotu, 0.35*d)
    th = np.radians(kat)
    obrot = lambda f: np.array([[np.cos(f), -np.sin(f)], [np.sin(f), np.cos(f)]])
    lewy = ostrze - dl*(obrot(th) @ e)
    prawy = ostrze - dl*(obrot(-th) @ e)
    return ([p0[0], ostrze[0], lewy[0], ostrze[0], prawy[0]],
            [p0[1], ostrze[1], lewy[1], ostrze[1], prawy[1]])

def przebieg_pociag(odl_x, odl_y, v_b, vp0, ap, a_b=(0, 0), T=T_MAX, N=60):
    """Sama fizyka (bez rysowania). Pszczoła startuje w (0; odl_y), ściana pociągu jest na y = 0.
    Okno ma lewy brzeg w x_okna(t) = odl_x + vp0*t + ap*t²/2 i szerokość SZER_OKNA."""
    v_b = np.asarray(v_b, float); a_b = np.asarray(a_b, float)
    A, B, C = a_b[1]/2, v_b[1], float(odl_y)          # y(t) = C + B*t + A*t² = 0
    if abs(A) < 1e-12:
        korzenie = [-C/B] if abs(B) > 1e-12 else []
    else:
        D = B**2 - 4*A*C
        korzenie = [(-B + np.sqrt(D))/(2*A), (-B - np.sqrt(D))/(2*A)] if D >= 0 else []
    dodatnie = [t for t in korzenie if t > 1e-9]
    t_ud = min(dodatnie) if dodatnie else None         # chwila dolotu do ściany
    dolecial = t_ud is not None and t_ud <= T
    t_end = t_ud if dolecial else T
    t = np.linspace(0, t_end, N + 1)
    w = dict(t=t, t_end=t_end,
             xb=v_b[0]*t + a_b[0]*t**2/2, yb=C + v_b[1]*t + a_b[1]*t**2/2,
             vxb=v_b[0] + a_b[0]*t, vyb=v_b[1] + a_b[1]*t,
             axb=np.full_like(t, a_b[0]), ayb=np.full_like(t, a_b[1]),
             xw=odl_x + vp0*t + ap*t**2/2, vp=vp0 + ap*t, ap=np.full_like(t, ap))
    if dolecial:
        w["yb"][-1] = 0.0
        dx = w["xb"][-1] - w["xw"][-1]                 # położenie pszczoły względem lewego brzegu okna
        if dx < 0:
            w["wynik"], w["roznica"] = "tyl", -dx
        elif dx > SZER_OKNA:
            w["wynik"], w["roznica"] = "przod", dx - SZER_OKNA
        else:
            w["wynik"], w["roznica"] = "trafiona", 0.0
    else:
        w["wynik"], w["roznica"] = "brak", None
    return w

def zakres_osi(*tablice):
    lo = min(float(np.min(x)) for x in tablice)
    hi = max(float(np.max(x)) for x in tablice)
    margines = 0.12*(hi - lo) if hi > lo else 0.5
    return [lo - margines, hi + margines]

def lot_do_okna(odleglosc_x, odleglosc_y, v_pszczoly, v_pociagu, a_pociagu, a_pszczoly=(0, 0)):
    """Animacja (widok z góry) i wykresy x(t), v(t), a(t). Pociąg jedzie w stronę rosnącego x."""
    if odleglosc_y <= 0:
        print("⚠️ odleglosc_y musi być większa od zera (pszczoła startuje przed ścianą pociągu).")
        return
    v_b = np.asarray(v_pszczoly, float); a_b = np.asarray(a_pszczoly, float)
    w = przebieg_pociag(odleglosc_x, odleglosc_y, v_b, v_pociagu, a_pociagu, a_b)
    t = w["t"]; N = len(t) - 1
    V = np.linalg.norm(v_b)

    v_max = float(np.max(np.hypot(w["vxb"], w["vyb"])))
    a_war = float(np.linalg.norm(a_b))
    sv = 1.0 if v_max <= 4 else 4/v_max            # skala strzałki prędkości [m na m/s]
    sa = 2.0 if a_war <= 2 else 4/a_war            # skala strzałki przyspieszenia [m na m/s²]

    def slady(k, pelne):
        def S(x, y, **styl):
            return go.Scatter(x=x, y=y, **(styl if pelne else {}))
        xo = w["xw"][k]; xl = xo - POL_OKNA
        poz = np.array([w["xb"][k], w["yb"][k]])
        vx_, vy_ = strzalka_2d(poz, sv*np.array([w["vxb"][k], w["vyb"][k]]))
        ax_, ay_ = strzalka_2d(poz, sa*np.array([w["axb"][k], w["ayb"][k]]))
        px_, py_ = strzalka_2d([xo + SZER_OKNA/2, -1.5], [sv*w["vp"][k], 0])
        ciche = dict(hoverinfo="skip", showlegend=False)
        return [
            S([xl, xl + DL_POCIAGU, xl + DL_POCIAGU, xl, xl], [-3, -3, 0, 0, -3], mode="lines", fill="toself",
              fillcolor="rgba(110,110,110,0.45)", line=dict(color="gray", width=2), **ciche),          # 0 pociąg
            S([xo, xo + SZER_OKNA, xo + SZER_OKNA, xo, xo], [-0.6, -0.6, 0, 0, -0.6], mode="lines",
              fill="toself", fillcolor="rgba(120,200,255,0.95)", line=dict(color="steelblue", width=2),
              **ciche),                                                                                  # 1 okno
            S(w["xb"][:k + 1], w["yb"][:k + 1], mode="lines",
              line=dict(color="gray", width=2, dash="dot"), **ciche),                                    # 2 ślad
            S([w["xb"][k]], [w["yb"][k]], mode="markers",
              marker=dict(size=18, color="#FFD400", line=dict(color="black", width=3)), **ciche),       # 3 pszczoła
            S(vx_, vy_, mode="lines", line=dict(color="royalblue", width=5), **ciche),                   # 4 v pszczoły
            S(ax_, ay_, mode="lines", line=dict(color=ZOLTY, width=6), **ciche),                         # 5 a pszczoły
            S(px_, py_, mode="lines", line=dict(color="black", width=4), **ciche),                       # 6 v pociągu
            S(t[:k + 1], w["xb"][:k + 1], mode="lines", line=dict(color="red", width=3),
              name="x pszczoły", showlegend=True),                                                      # 7
            S(t[:k + 1], w["yb"][:k + 1], mode="lines", line=dict(color="red", width=3, dash="dash"),
              name="y pszczoły", showlegend=True),                                                      # 8
            S(t[:k + 1], w["xw"][:k + 1], mode="lines", line=dict(color="gray", width=4),
              name="x okna (pociąg)", showlegend=True),                                                 # 9
            S(t[:k + 1], w["vxb"][:k + 1], mode="lines", line=dict(color="royalblue", width=3),
              name="vx pszczoły", showlegend=True),                                                     # 10
            S(t[:k + 1], w["vyb"][:k + 1], mode="lines", line=dict(color="royalblue", width=3, dash="dash"),
              name="vy pszczoły", showlegend=True),                                                     # 11
            S(t[:k + 1], w["vp"][:k + 1], mode="lines", line=dict(color="gray", width=4),
              name="v pociągu", showlegend=True),                                                       # 12
            S(t[:k + 1], w["axb"][:k + 1], mode="lines", line=dict(color=ZOLTY, width=3),
              name="ax pszczoły", showlegend=True),                                                     # 13
            S(t[:k + 1], w["ayb"][:k + 1], mode="lines", line=dict(color=ZOLTY, width=3, dash="dash"),
              name="ay pszczoły", showlegend=True),                                                     # 14
            S(t[:k + 1], w["ap"][:k + 1], mode="lines", line=dict(color="gray", width=4),
              name="a pociągu", showlegend=True),                                                       # 15
        ]

    wiersze = [1]*7 + [2]*3 + [3]*3 + [4]*3
    fig = make_subplots(rows=4, cols=1, row_heights=[0.40, 0.20, 0.20, 0.20], vertical_spacing=0.06)
    for tr, wiersz in zip(slady(0, True), wiersze):
        fig.add_trace(tr, row=wiersz, col=1)
    fig.add_trace(go.Scatter(x=[0], y=[odleglosc_y], mode="markers+text", text=["start"],     # 16 start
                             textposition="top center", marker=dict(size=6, color="gray"),
                             hoverinfo="skip", showlegend=False), row=1, col=1)
    fig.frames = [go.Frame(data=slady(k, False), traces=list(range(16)), name=str(k))
                  for k in range(N + 1)]

    # zakresy osi
    x_wszystko = np.concatenate([w["xb"], w["xw"] - POL_OKNA, w["xw"] - POL_OKNA + DL_POCIAGU, [0.0]])
    zx = [float(x_wszystko.min()) - 2, float(x_wszystko.max()) + 2]
    zy = [min(-3.5, float(w["yb"].min()) - 1), max(float(w["yb"].max()), float(odleglosc_y)) + 1.5]
    fig.update_xaxes(title_text="x [m]", range=zx, constrain="domain", row=1, col=1)
    fig.update_yaxes(title_text="y [m]", range=zy, scaleanchor="x", scaleratio=1,
                     constrain="domain", row=1, col=1)
    opisy = {2: ("położenie [m]", (w["xb"], w["yb"], w["xw"])),
             3: ("prędkość [m/s]", (w["vxb"], w["vyb"], w["vp"])),
             4: ("przyspieszenie [m/s²]", (w["axb"], w["ayb"], w["ap"]))}
    for wiersz, (napis, tablice) in opisy.items():
        fig.update_xaxes(range=[0, w["t_end"]], title_text="t [s]" if wiersz == 4 else None,
                         row=wiersz, col=1)
        fig.update_yaxes(title_text=napis, range=zakres_osi(*tablice), row=wiersz, col=1)

    czasy = t
    suwak, odtwarzanie = kontrolki_animacji(czasy, int(1000*min(w["t_end"], 8)/N), False, 60)
    grupy = [("Wektor v", [4], {}, {}), ("Wektor a", [5], {}, {}), ("v pociągu", [6], {}, {})]
    tytul = ("<b>Pszczoła i okno pociągu</b> (widok z góry, pociąg jedzie w prawo)"
             f"<br><sup><span style=\"color:royalblue\"><b>niebieski</b></span> – v pszczoły (1 m/s = {sv:.2g} m),   "
             f"<span style=\"color:{ZOLTY}\"><b>żółty</b></span> – a pszczoły (1 m/s² = {sa:.2g} m),   "
             "<b>czarny</b> – v pociągu</sup>")
    fig.update_layout(title=dict(text=tytul, x=0.5, y=0.99, yanchor="top"), height=1150,
                      margin=dict(l=70, r=170, t=130, b=170),
                      updatemenus=menu_przelacznikow(grupy, 1.0) + [odtwarzanie], sliders=[suwak],
                      legend=dict(x=1.01, y=0.5, xanchor="left"), uirevision="stala")

    # werdykt
    print(f"Pszczoła: v₀ = {np.round(v_b, 2)} m/s (|v₀| = {V:.2f} m/s),  a = {np.round(a_b, 2)} m/s²")
    print(f"Pociąg:   v₀ = {v_pociagu:g} m/s,  a = {a_pociagu:g} m/s²")
    za_szybko = V > V_MAX_PSZCZOLY + 1e-9
    if za_szybko:
        print(f"⛔ Limit: prędkość początkowa pszczoły może wynosić najwyżej {V_MAX_PSZCZOLY:g} m/s – ten lot się nie liczy.")
    wynik = w["wynik"]
    if wynik == "brak":
        print(f"🐝 Pszczoła nie dolatuje do ściany pociągu w ciągu {T_MAX:g} s.")
    else:
        xo = w["xw"][-1]
        print(f"Pszczoła dolatuje do ściany po t = {w['t_end']:.2f} s, w punkcie x = {w['xb'][-1]:.2f} m "
              f"(okno zajmuje wtedy {xo:.2f} … {xo + SZER_OKNA:.2f} m).")
        if wynik == "trafiona":
            print("🎉 Pszczoła wleciała do okna!" if not za_szybko else "(Trafiła w okno, ale przekroczyła limit prędkości.)")
        elif wynik == "tyl":
            print(f"💥 Pszczoła uderza w ścianę {w['roznica']:.2f} m za oknem – okno uciekło jej do przodu.")
        else:
            print(f"💥 Pszczoła uderza w ścianę {w['roznica']:.2f} m przed oknem – wyprzedziła je.")
    fig.show()

def czy_pszczola_da_rade(odleglosc_x, odleglosc_y, v_pociagu, a_pociagu, v_max=V_MAX_PSZCZOLY):
    """Sprawdza wiele możliwych prędkości początkowych pszczoły (bez jej przyspieszenia, |v| <= v_max).
    Zwraca (True, przykładowa prędkość) albo (False, None)."""
    kat = np.radians(np.linspace(181, 359, 357))        # kierunki lotu w stronę ściany (vy < 0)
    szybkosc = np.linspace(0.2, v_max, 80)
    S, K = np.meshgrid(szybkosc, kat)
    vx, vy = S*np.cos(K), S*np.sin(K)
    t = odleglosc_y / (-vy)                              # czas dolotu do ściany
    roznica = vx*t - (odleglosc_x + v_pociagu*t + a_pociagu*t**2/2)
    dobre = (roznica >= 0) & (roznica <= SZER_OKNA) & (t <= T_MAX)
    if not dobre.any():
        return False, None
    i = np.argmin(np.where(dobre, t, np.inf))            # wybieramy najkrótszy lot
    return True, np.round(np.array([vx.flat[i], vy.flat[i]]), 2)

## 1. Przyspieszenie

Prędkość to szybkość zmiany **położenia**. Przyspieszenie to szybkość zmiany **prędkości**. Wzór jest bliźniaczo podobny do tego z części 2.

Pszczoła w chwili $t_1$ ma prędkość $\vec{v}_1$, a w chwili $t_2$ prędkość $\vec{v}_2$. **Zmiana prędkości** to:

$$\Delta\vec{v} = \vec{v}_2 - \vec{v}_1$$

**Przyspieszenie** (średnie) to zmiana prędkości przypadająca na jednostkę czasu:

$$\vec{a} = \frac{\Delta\vec{v}}{\Delta t}$$

Jednostką jest **metr na sekundę do kwadratu** (m/s²). Przyspieszenie 2 m/s² oznacza, że **co sekundę** prędkość zmienia się o 2 m/s.

### Przyspieszenie to wektor, a nie tylko „szybciej"
Prędkość $\vec{v}$ jest wektorem, więc zmienia się, gdy zmienia się jej **wartość** *albo* **kierunek**. Wektor $\vec{a}$ jest skierowany tak samo jak $\Delta\vec{v}$, czyli tak, w którą stronę „ciągnie" prędkość:

- $\vec{a}$ **zgodne z** $\vec{v}$ → pszczoła **przyspiesza** (leci coraz szybciej),
- $\vec{a}$ **przeciwne do** $\vec{v}$ → pszczoła **hamuje**,
- $\vec{a}$ **prostopadłe do** $\vec{v}$ → pszczoła **skręca** (zmienia kierunek lotu).

In [ ]:
v1 = np.array([1, 0.5, 0])    # prędkość na początku [m/s]
v2 = np.array([3, 1.5, 0])    # prędkość po czasie dt [m/s]
dt = 4                        # czas [s]

dv = v2 - v1                  # zmiana prędkości
a = dv / dt                   # przyspieszenie

print("Zmiana prędkości Δv =", dv, "m/s")
print("Przyspieszenie a =", a, "m/s²")
print(f"Wartość przyspieszenia |a| = {np.linalg.norm(a):.2f} m/s²")

> 💡 **Uwaga o Pythonie:** to dokładnie ta sama konstrukcja co w części 2, tylko zamiast położeń odejmujemy prędkości. Wektory `np.array` odejmuje się i dzieli przez liczbę **składowa po składowej**.

> ✏️ **Twój kod: hamowanie.** Pszczoła leci z prędkością $\vec{v}_1 = (6;\ 0;\ 0)$ m/s. Po 4 s jej prędkość to $\vec{v}_2 = (2;\ 0;\ 0)$ m/s. **Napisz kod**, który obliczy przyspieszenie. Jaki ma znak? Co to oznacza?

In [ ]:
v1 = np.array([6, 0, 0])
v2 = np.array([2, 0, 0])
dt = 4

# Twój kod tutaj:

<details>
<summary>💡 Rozwiązanie</summary>

```python
a = (v2 - v1) / dt
print("a =", a, "m/s²")
```

Wynik: $\vec{a} = (-1;\ 0;\ 0)$ m/s². Minus oznacza, że przyspieszenie jest skierowane **przeciwnie** do prędkości (która jest w kierunku $+x$), czyli pszczoła **hamuje**: co sekundę traci 1 m/s.
</details>

## 2. Ruch ze stałym przyspieszeniem

Załóżmy, że przyspieszenie $\vec{a}$ jest **stałe**. Przekształcając wzór na przyspieszenie, dostajemy prędkość w chwili $t$ (liczonej od startu, $\vec{v}_0$ to prędkość początkowa):

$$\vec{v}(t) = \vec{v}_0 + \vec{a}\,t$$

A położenie? Prędkość rośnie **równomiernie**, więc jej średnia w czasie $t$ to średnia z prędkości początkowej i końcowej: $\frac{\vec{v}_0 + \vec{v}}{2} = \vec{v}_0 + \frac{1}{2}\vec{a}\,t$. Przemieszczenie to prędkość średnia razy czas, więc:

$$\vec{r}(t) = \vec{r}_0 + \vec{v}_0\,t + \frac{1}{2}\vec{a}\,t^2$$

Dla $\vec{a} = 0$ wracamy do wzoru z części 2: $\vec{r} = \vec{r}_0 + \vec{v}\,t$.

In [ ]:
r0 = np.array([-4, -2, 0])      # położenie początkowe [m]
v0 = np.array([1, 0.5, 0])      # prędkość początkowa [m/s]
a  = np.array([0.5, 0.25, 0])   # przyspieszenie [m/s²]
t  = 3                          # czas [s]

v = v0 + a * t
r = r0 + v0 * t + a * t**2 / 2

print(f"Po {t} s: prędkość v = {v} m/s")
print(f"Po {t} s: położenie r = {r} m")

## 3. Obserwacja: jak lata pszczoła z przyspieszeniem?

Zobaczmy to na wykresie. Do znanych już wektorów dochodzi **żółty wektor przyspieszenia** $\vec{a}$:

- **czerwony** – położenie $\vec{r}$, **niebieski** – prędkość $\vec{v}$, **żółty** – przyspieszenie $\vec{a}$ (oba ostatnie przyczepione do pszczoły),
- skale strzałek są podane pod tytułem wykresu (żeby wszystko było czytelne),
- **▶ Start** puszcza lot, suwak czasu pozwala go przewijać, wykres możesz obracać,
- przyciski *pokaż / ukryj* pozwalają ukrywać elementy (także kilka naraz). Przerywana linia to cały tor lotu.

In [ ]:
r0 = np.array([-4, -2, 0])      # położenie początkowe [m]
v0 = np.array([1, 0.5, 0])      # prędkość początkowa [m/s]
a  = np.array([0.5, 0.25, 0])   # przyspieszenie [m/s²]
T  = 5                          # czas lotu [s]

lot_z_przyspieszeniem(r0, v0, a, T)

### 🔬 Eksperymenty
Wpisz podane liczby do komórki wyżej i uruchom ją ponownie. Najpierw **przewiduj**, potem sprawdzaj.

**A. Przyspieszenie zgodne z prędkością** (to, co jest teraz w komórce). Obserwuj niebieski wektor w czasie lotu. Czy żółty zmienia długość i kierunek? Jak tor wygląda w porównaniu z lotem po prostej?

**B. Hamowanie.** `v0 = [3, 1.5, 0]`, `a = [-0.5, -0.25, 0]`, `T = 10`. W której chwili prędkość jest równa zero? (Policz: kiedy $\vec{v}_0 + \vec{a}\,t = 0$?) Co dzieje się później i w którą stronę leci pszczoła?

**C. Skręcanie.** `r0 = [-4, -2, 0]`, `v0 = [2, 0, 0]`, `a = [0, 0.3, 0]`, `T = 6`. Przyspieszenie jest **prostopadłe** do prędkości w chwili startu. Co dzieje się z kierunkiem lotu? Czy tor jest prostą?

**D. Grawitacja.** `r0 = [0, 0, 0]`, `v0 = [4, 2, 6]`, `a = [0, 0, -9.81]`, `T = 1.2`. Ziemska grawitacja to też przyspieszenie, skierowane w dół ($g \approx 9{,}81$ m/s²). Znajomy kształt toru? Do tego wrócimy w kolejnej lekcji: to **rzut ukośny**!

> 🤔 **Pytania.** (1) Czy wektor $\vec{a}$ musi być zawsze skierowany tak jak $\vec{v}$? (2) Czy pszczoła może się poruszać, mając $\vec{a} = 0$? (3) Czy może mieć $\vec{v} = 0$ i $\vec{a} \neq 0$ w tej samej chwili? (Podpowiedź: eksperyment B.)

## 4. 🚆 Zadanie: pszczoła kontra pociąg

Teraz poruszamy się w **dwóch wymiarach** (widok z góry). Pociąg stoi przy torze i **powoli rusza**. W jego ścianie jest okno o szerokości 1,2 m. Pszczoła musi wlecieć do środka.

**Model:**
- Tor biegnie wzdłuż osi $x$, pociąg jedzie w stronę rosnącego $x$. Ściana pociągu leży na prostej $y = 0$.
- **Pszczoła** startuje w punkcie $(0;\ \text{odleglosc\_y})$ i leci ze stałą prędkością $\vec{v}$ (opcjonalnie z własnym przyspieszeniem). Żeby podlecieć do ściany, musi mieć $v_y < 0$.
- **Okno**: w chwili startu jego lewy brzeg jest w $x = \text{odleglosc\_x}$ i porusza się ze stałym przyspieszeniem pociągu:

$$x_{okna}(t) = \text{odleglosc\_x} + v_{p0}\,t + \tfrac{1}{2}\,a_p\,t^2$$

- Pszczoła **wleci do okna**, jeśli dotrze do ściany ($y = 0$) w chwili, gdy jej położenie $x$ mieści się między lewym i prawym brzegiem okna.
- Pszczoła jest punktem. Jej **prędkość początkowa** nie może przekraczać **4 m/s**.

**Twoje zadanie:** dobierz `v_pszczoly` (czyli $v_x$ i $v_y$), żeby pszczoła wleciała do okna. Możesz też zmieniać odległości oraz prędkość i przyspieszenie pociągu.

Pod animacją **powstają wykresy** położenia, prędkości i przyspieszenia od czasu: pszczoły (kolor) i pociągu (szary). Włącz ▶ i obserwuj.

In [ ]:
# ----------- ZMIENIAJ TE LICZBY -----------
odleglosc_x = 6            # lewy brzeg okna jest tyle metrów przed pszczołą (wzdłuż toru) [m]
odleglosc_y = 6            # pszczoła jest tyle metrów od ściany pociągu [m]
v_pszczoly  = [1.5, -2]    # prędkość początkowa pszczoły [vx, vy] w m/s
v_pociagu   = 0            # prędkość początkowa pociągu [m/s]
a_pociagu   = 0.3          # przyspieszenie pociągu [m/s²]
# ------------------------------------------

lot_do_okna(odleglosc_x, odleglosc_y, v_pszczoly, v_pociagu, a_pociagu)

### Jak czytać wykresy?
- **Położenie.** Krzywa $x$ pszczoły (czerwona) i $x$ okna (szara). Pszczoła trafia w okno, gdy w chwili dolotu do ściany czerwona linia leży **na szarej** (lub w jej szerokości okna). Przerywana czerwona linia to $y$, spadająca do zera przy ścianie.
- **Prędkość.** Pszczoła leci ze stałą prędkością (poziome linie). Prędkość pociągu rośnie **liniowo**. Nachylenie tej linii to przyspieszenie pociągu.
- **Przyspieszenie.** Stałe (poziome linie). Żółte linie są zerowe, bo pszczoła nie przyspiesza.
- **Związek wykresów:** nachylenie wykresu położenia to prędkość, a nachylenie wykresu prędkości to przyspieszenie.

<details>
<summary>💡 Podpowiedź 1: jak do tego podejść?</summary>

Najpierw pomyśl, **kiedy** pszczoła doleci do ściany. Odległość `odleglosc_y` pokonuje ona z prędkością $|v_y|$, więc $t = \text{odleglosc\_y}/|v_y|$. Następnie sprawdź, **gdzie będzie okno** w tej chwili, ze wzoru na $x_{okna}(t)$. Pszczoła musi mieć wtedy $x = v_x\,t$ równe położeniu okna.
</details>

<details>
<summary>💡 Podpowiedź 2: gotowy sposób w Pythonie</summary>

```python
vy = -2
t_dolotu = odleglosc_y / abs(vy)
x_okna = odleglosc_x + v_pociagu * t_dolotu + a_pociagu * t_dolotu**2 / 2
vx = (x_okna + 0.6) / t_dolotu          # 0.6 = połowa szerokości okna (trafiamy w środek)
v_pszczoly = [vx, vy]
print(v_pszczoly)
```

Dla domyślnych liczb wychodzi $v_x \approx 2{,}65$ m/s, więc $|\vec{v}| \approx 3{,}3$ m/s. To mieści się w limicie 4 m/s.
</details>

> 🤔 **Pytania.** Czy rozwiązanie jest tylko jedno? Co się stanie, gdy zwiększysz `vy` (pszczoła podleci szybciej do ściany)? Co zmienia się na wykresach, gdy pociąg rusza z większą prędkością początkową?

## 5. 🔍 Badanie: kiedy pszczoła nie ma szans?

Zauważyłeś, że im szybciej jedzie pociąg, tym trudniej go „trafić"? Pszczoła ma **limit 4 m/s**, a okno ucieka. Sprawdźmy to systematycznie.

Funkcja `czy_pszczola_da_rade(...)` przeszukuje wiele możliwych prędkości początkowych pszczoły (o wartości do 4 m/s) i odpowiada, czy **jakakolwiek** z nich prowadzi do okna. Użyjemy jej w pętli, żeby sprawdzić kolejne prędkości pociągu:

In [ ]:
odleglosc_x = 6
odleglosc_y = 6
a_pociagu = 0.3

for v_pociagu in [0, 1, 2, 3, 4, 5]:
    da_rade, przyklad = czy_pszczola_da_rade(odleglosc_x, odleglosc_y, v_pociagu, a_pociagu)
    if da_rade:
        print(f"v pociągu = {v_pociagu} m/s  ->  TAK, np. v pszczoły = {przyklad} m/s")
    else:
        print(f"v pociągu = {v_pociagu} m/s  ->  NIE, pszczoła nie ma szans")

> 💡 **Uwaga o Pythonie:** `for ... in [...]:` powtarza kod dla każdej wartości z listy, a `if ... else` wybiera, co wypisać. Pamiętaj o **wcięciach** (4 spacje): Python po nich rozpoznaje, co należy do pętli.

> ✏️ **Zadanie 2 (badanie).** Zmieniaj parametry w komórce wyżej i znajdź odpowiedzi:
> 1. Przy jakiej mniej więcej prędkości początkowej pociągu pszczoła przestaje mieć szanse? Zawęź przedział, wpisując np. `[1.5, 1.6, 1.7, 1.8, 1.9, 2.0]`.
> 2. Co się zmieni, jeśli pszczoła zacznie bliżej ściany (`odleglosc_y = 3`)? A jeśli dalej (`odleglosc_y = 10`)?
> 3. Co zmieni się, gdy pociąg przyspiesza mocniej (`a_pociagu = 1`)?
> 4. Dlaczego bliższa ściana pomaga, a dalsza szkodzi? (Pomyśl, co dzieje się z oknem przez dłuższy czas lotu.)
> 5. Sprawdź przykładową prędkość z wyniku na animacji, wpisując ją do `lot_do_okna(...)`. Czy się zgadza?

<details>
<summary>💡 Odpowiedź (kliknij po własnych próbach)</summary>

Dla danych z komórki (`odleglosc_x = 6`, `odleglosc_y = 6`, `a_pociagu = 0.3`) pszczoła ma szanse tylko dla $v_{p0}$ mniejszych niż około **2 m/s**. Dla szybszego pociągu okno ucieka szybciej, niż pszczoła (przy limicie 4 m/s) może je dogonić w czasie lotu do ściany. Im dalej ściana, tym dłużej lot i tym dalej odjedzie okno, więc tym trudniej. Przy `odleglosc_y = 3` pszczoła radzi sobie z szybszym pociągiem.
</details>

### 🚀 Wyzwanie: pszczoła z silniczkiem
Dla pociągu o prędkości początkowej 5 m/s pszczoła ze stałą prędkością nie ma szans. Ale pszczoła też może **przyspieszać**! Dodaj jej przyspieszenie `a_pszczoly = [ax, ay]` i dobierz je tak, żeby wleciała do okna. (Limit 4 m/s dotyczy tylko prędkości początkowej.)

In [ ]:
odleglosc_x = 6
odleglosc_y = 6
v_pszczoly  = [2, -2]       # prędkość początkowa pszczoły [m/s]
v_pociagu   = 5             # za szybki pociąg dla pszczoły bez przyspieszenia
a_pociagu   = 0.3
a_pszczoly  = [0, 0]        # <-- zmień (ax, ay) [m/s²]; żółty wektor pojawi się na wykresie

lot_do_okna(odleglosc_x, odleglosc_y, v_pszczoly, v_pociagu, a_pociagu, a_pszczoly)

<details>
<summary>💡 Podpowiedź</summary>

Zostaw $a_y = 0$, wtedy $t_{dolotu} = 6/2 = 3$ s. W tym czasie okno przejedzie do $x_{okna}(3) = 6 + 5\cdot 3 + \tfrac{1}{2}\cdot 0{,}3\cdot 3^2 = 22{,}35$ m. Pszczoła ma być w środku okna: $x = 22{,}35 + 0{,}6 = 22{,}95$ m. Ze wzoru $x = v_x t + \tfrac{1}{2}a_x t^2$ dostajemy:

```python
t = 3
a_x = 2 * (22.95 - 2 * t) / t**2      # ok. 3.77 m/s²
a_pszczoly = [a_x, 0]
```
</details>

## 6. Podsumowanie

**Zmiana prędkości** (w Pythonie: `dv = v2 - v1`)

$$\Delta\vec{v} = \vec{v}_2 - \vec{v}_1$$

**Przyspieszenie – wektor** (w Pythonie: `a = dv / dt`), jednostka m/s²

$$\vec{a} = \frac{\Delta\vec{v}}{\Delta t}$$

**Prędkość przy stałym przyspieszeniu** (w Pythonie: `v = v0 + a * t`)

$$\vec{v}(t) = \vec{v}_0 + \vec{a}\,t$$

**Położenie przy stałym przyspieszeniu** (w Pythonie: `r = r0 + v0 * t + a * t**2 / 2`)

$$\vec{r}(t) = \vec{r}_0 + \vec{v}_0\,t + \frac{1}{2}\vec{a}\,t^2$$

**Najważniejsze:**
- przyspieszenie jest wektorem: zmienia się, gdy zmienia się **wartość lub kierunek** prędkości,
- $\vec{a}$ zgodne z $\vec{v}$ → przyspieszanie, przeciwne → hamowanie, prostopadłe → skręcanie,
- nachylenie wykresu $x(t)$ to prędkość, a nachylenie wykresu $v(t)$ to przyspieszenie.

**W następnej części:** rzut ukośny. Grawitacja to stałe przyspieszenie skierowane w dół, więc wszystko, co umiesz, wystarczy do opisania lotu piłki.

---
### 🔎 Dla ciekawskich
Funkcja `lot_do_okna` liczy w kodzie pomocniczym, **kiedy** pszczoła dotrze do ściany. Rozwiązuje równanie $y(t) = 0$, czyli zwykłe równanie kwadratowe, gdy pszczoła ma przyspieszenie w kierunku $y$. Potem sprawdza, czy w tej chwili jej $x$ mieści się w oknie. Funkcja `czy_pszczola_da_rade` robi to samo dla tysięcy prędkości początkowych naraz (wszystkie „próby" są jedną operacją na tablicach `numpy`).

---

[← Lekcja 2: Prędkość](https://colab.research.google.com/github/InteraktywnaFizykaZPytonem/Interaktywna-Fizyka-z-Pytonem/blob/main/Kinematyka_2_predkosc.ipynb) &nbsp;·&nbsp; [📚 Spis lekcji](https://github.com/InteraktywnaFizykaZPytonem/Interaktywna-Fizyka-z-Pytonem#spis-lekcji) &nbsp;·&nbsp; **Lekcja 3 z 3** (następne wkrótce)